# 스마트 창고 출고 지연 예측 — 17위 솔루션 (Public 9.7527 / Private 10.01261)

**대회**: 데이콘 스마트 창고 출고 지연 예측 경진대회  
**결과**: 608팀 중 17위 (Public LB 9.7527 / Private LB 10.01261)  
**핵심 접근**: v31 피처(335개) × Mega33 스태킹 × Sequential Oracle 블렌드

---

## 전체 파이프라인 요약

```
원시 데이터
    ↓
[피처 엔지니어링 v31: 335개]
    ↓
[Base 모델: LGB + XGB + CatBoost × GroupKFold×5]
    ↓
[Mega33 스태킹: 33개 모델 OOF → Meta LGB/XGB/CB 평균]
    ↓
[Sequential Oracle: mega33 OOF를 lag 피처로 재학습]
    ↓
[최종 블렌드: FIXED = mega33×0.7637 + rank_adj×0.1589 + iter_r1~r3
oracle_NEW = 0.64×FIXED + 0.12×oracle_xgb + 0.16×oracle_lv2 + 0.08×oracle_rem]
    ↓
Public LB 9.7527
```


## Section 1: 데이터 로드 & 문제 파악


In [ ]:
import numpy as np, pandas as pd, pickle, os, time, warnings, gc
warnings.filterwarnings('ignore')
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error

TARGET = 'avg_delay_minutes_next_30m'
N_SPLITS = 5
SEED = 42

train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')
sample = pd.read_csv('sample_submission.csv')

# row_id 정렬 (rid order)
train['_row_id'] = train['ID'].str.replace('TRAIN_','').astype(int)
test['_row_id']  = test['ID'].str.replace('TEST_','').astype(int)
train = train.sort_values('_row_id').reset_index(drop=True)
test  = test.sort_values('_row_id').reset_index(drop=True)

print(f"train: {train.shape}, test: {test.shape}")
print(f"target: mean={train[TARGET].mean():.2f}, std={train[TARGET].std():.2f}, max={train[TARGET].max():.2f}")

### 데이터 구조 정리

| 항목 | 내용 |
|------|------|
| 타겟 | `avg_delay_minutes_next_30m` (향후 30분 평균 출고 지연 분) |
| 그룹 단위 | `layout_id` × `scenario_id` (창고 레이아웃 × 시뮬레이션 시나리오) |
| 시계열 구조 | 각 (layout, scenario) 내에서 시간 순으로 관측값 나열 |
| 핵심 특이사항 | **test 셋에 train에 없는 layout_id(unseen)가 존재** → 도메인 시프트 문제 |

> **주의**: `ID` 컬럼의 숫자가 layout/scenario 순서가 아닌 row 순서임.  
> 피처 엔지니어링 시 `(layout_id, scenario_id)` 기준으로 정렬(ls order)하고,  
> 최종 제출 시 `ID` 기준(rid order)으로 되돌려야 한다.


## Section 2: EDA — Seen/Unseen 도메인 시프트 (핵심 발견)


In [ ]:
train_layouts = set(train['layout_id'].unique())
test_layouts  = set(test['layout_id'].unique())
seen_test   = test_layouts & train_layouts
unseen_test = test_layouts - train_layouts

print(f"train layouts: {len(train_layouts)}")
print(f"test layouts: {len(test_layouts)}  (seen: {len(seen_test)}, unseen: {len(unseen_test)})")

# 핵심 피처 분포 비교
key_feats = ['order_inflow_15m', 'pack_utilization', 'congestion_score', 'robot_utilization']
train_mean = train[key_feats].mean()
test_seen_mean   = test[test['layout_id'].isin(seen_test)][key_feats].mean()
test_unseen_mean = test[test['layout_id'].isin(unseen_test)][key_feats].mean()

comparison = pd.DataFrame({'train': train_mean, 'test_seen': test_seen_mean, 'test_unseen': test_unseen_mean})
comparison['seen_ratio']   = comparison['test_seen']   / comparison['train']
comparison['unseen_ratio'] = comparison['test_unseen'] / comparison['train']
print(comparison.round(3))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, feat in enumerate(key_feats):
    ax = axes[i]
    ax.hist(train[feat].dropna(), bins=50, alpha=0.5, label='train', density=True)
    ax.hist(test[test['layout_id'].isin(seen_test)][feat].dropna(),
            bins=50, alpha=0.5, label='test_seen', density=True)
    ax.hist(test[test['layout_id'].isin(unseen_test)][feat].dropna(),
            bins=50, alpha=0.5, label='test_unseen', density=True)
    ax.set_title(feat)
    ax.legend(fontsize=7)
plt.suptitle('Seen vs Unseen 도메인 시프트 시각화', fontsize=13)
plt.tight_layout()
plt.show()

### 도메인 시프트 핵심 인사이트

- **test_unseen** 레이아웃은 train에 전혀 등장하지 않은 창고 구조
- `congestion_score`, `pack_utilization` 등에서 분포 차이가 존재
- **시도했지만 실패한 접근**: seen/unseen을 분기하여 별도 모델 최적화 (Gate 계열)  
  → OOF에서 개선되어도 LB에서 악화됨 (과적합)
- **효과적인 접근**: GroupKFold(layout_id) → unseen layout이 val에 들어가는 fold를 자연스럽게 처리


## Section 3: 피처 엔지니어링 (v31, 335개)

### 피처 카테고리 요약

| 카테고리 | 개수 | 설명 |
|----------|------|------|
| Raw 원시 피처 | 43개 | train 컬럼 그대로 사용 |
| Lag 피처 | 22개 | lag1~lag3, within scenario (이전 타임스텝 값) |
| Lead 피처 | 30개 | lead1~lead5, within scenario (다음 타임스텝 값, train only) |
| Rolling 피처 | 105개 | SC mean/std/max/min, rolling window (scenario 내 누적 통계) |
| Scenario 집계 | 49개 | 시나리오 전체 통계 (그룹 수준 집계) |
| 기타 | 86개 | diff, ratio, interaction 피처 |
| **합계** | **335개** | |

> Lead 피처는 train에서만 생성 가능 (미래 정보 활용). test에서는 NaN 처리 또는 global mean으로 대체.


In [ ]:
# ─── 인덱스 정렬 ───────────────────────────────────────────────
# 피처 캐시는 (layout_id, scenario_id) 정렬 순서(ls order)로 저장
train_ls = pd.read_csv('train.csv').sort_values(['layout_id','scenario_id']).reset_index(drop=True)
test_ls  = pd.read_csv('test.csv').sort_values(['layout_id','scenario_id']).reset_index(drop=True)

ls_pos    = {row['ID']:i for i,row in train_ls.iterrows()}
te_ls_pos = {row['ID']:i for i,row in test_ls.iterrows()}
id2    = np.array([ls_pos[i]    for i in train['ID'].values])   # rid→ls 변환 인덱스
te_id2 = np.array([te_ls_pos[i] for i in test['ID'].values])

# ─── 피처 로드 ────────────────────────────────────────────────
with open('results/eda_v31/v31_fe_cache.pkl', 'rb') as f:
    fe = pickle.load(f)

feat_cols = fe['feat_cols']
train_fe  = fe['train_fe']   # ls order
test_fe   = fe['test_fe']    # ls order
del fe; gc.collect()

y      = train_ls[TARGET].values.astype(np.float64)
y_log  = np.log1p(y)
X_tr   = train_fe[feat_cols].values.astype(np.float32)   # ls order
X_te   = test_fe[[c for c in feat_cols if c in test_fe.columns]].values.astype(np.float32)
groups = train_ls['layout_id'].values

print(f"피처 수: {len(feat_cols)}")
print(f"X_tr: {X_tr.shape}, X_te: {X_te.shape}")

In [ ]:
# ─── 피처 엔지니어링 핵심 로직 (요약) ─────────────────────────
# 실제로는 results/eda_v31/v31_fe_cache.pkl 에 캐시되어 있음

def build_features_v31(df, is_train=True):
    """v31 피처 엔지니어링 핵심 로직"""
    df = df.sort_values(['layout_id','scenario_id']).reset_index(drop=True)
    grp = df.groupby(['layout_id','scenario_id'])

    # 1) Lag 피처 (within scenario)
    lag_cols = ['order_inflow_15m', 'pack_utilization', 'congestion_score',
                'robot_utilization', 'battery_mean', 'fault_count_15m',
                'blocked_path_15m', 'charge_queue_length']
    for col in lag_cols:
        for lag in [1, 2, 3]:
            df[f'{col}_lag{lag}'] = grp[col].shift(lag)

    # 2) Lead 피처 (train only — 미래 정보)
    if is_train:
        for col in lag_cols[:5]:
            for lead in [1, 2, 3, 4, 5]:
                df[f'{col}_lead{lead}'] = grp[col].shift(-lead)

    # 3) Rolling 피처 (누적 이동 통계)
    roll_cols = ['order_inflow_15m', 'pack_utilization', 'congestion_score']
    for col in roll_cols:
        for window in [3, 5, 10]:
            rolled = grp[col].rolling(window, min_periods=1)
            df[f'{col}_roll{window}_mean'] = rolled.mean().values
            df[f'{col}_roll{window}_std']  = rolled.std().values
            df[f'{col}_roll{window}_max']  = rolled.max().values
            df[f'{col}_roll{window}_min']  = rolled.min().values

    # 4) Scenario 집계 피처 (그룹 전체 통계)
    sc_agg = grp[lag_cols].agg(['mean','std','max','min'])
    sc_agg.columns = [f'sc_{c}_{f}' for c,f in sc_agg.columns]
    sc_agg = sc_agg.reset_index()
    df = df.merge(sc_agg, on=['layout_id','scenario_id'], how='left')

    # 5) Interaction 피처
    df['congestion_x_inflow'] = df['congestion_score'] * df['order_inflow_15m']
    df['pack_x_robot']        = df['pack_utilization'] * df['robot_utilization']
    df['inflow_diff1']        = df['order_inflow_15m'] - df.get('order_inflow_15m_lag1', df['order_inflow_15m'])
    df['pack_ratio_sc']       = df['pack_utilization'] / (df.get('sc_pack_utilization_mean', 1) + 1e-6)

    return df

print("피처 엔지니어링 함수 정의 완료")
print("실제 실행은 캐시(v31_fe_cache.pkl)에서 로드")

## Section 4: Base 모델 학습 (LGB + XGB + CatBoost, GroupKFold×5)

### 교차검증 전략

- **GroupKFold(layout_id)**: 동일 창고 레이아웃이 train/val에 섞이지 않도록
- val fold가 unseen layout을 자연스럽게 시뮬레이션
- 타겟 변환: `log1p(y)` → 예측 후 `expm1()` 역변환
- Early Stopping: 100 rounds (과적합 방지)


In [ ]:
gkf   = GroupKFold(n_splits=N_SPLITS)
folds = list(gkf.split(np.arange(len(y)), groups=groups))

# fold 번호 저장 (이후 스태킹에서 재사용)
fold_ids = np.zeros(len(y), dtype=int)
for f_i, (_, val_idx) in enumerate(folds):
    fold_ids[val_idx] = f_i

os.makedirs('results/base_v31', exist_ok=True)

print(f"GroupKFold splits:")
for f_i, (tr_idx, val_idx) in enumerate(folds):
    val_layouts = set(groups[val_idx])
    seen_in_val = val_layouts & train_layouts if 'train_layouts' in dir() else set()
    print(f"  Fold {f_i+1}: train={len(tr_idx)}, val={len(val_idx)}, val_layouts={len(val_layouts)}")

In [ ]:
# ─── LightGBM ─────────────────────────────────────────────────
LGB_PARAMS = dict(
    objective='mae', n_estimators=3000, learning_rate=0.03,
    num_leaves=63, max_depth=6, min_child_samples=50,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.5, reg_lambda=0.5,
    random_state=SEED, verbose=-1, n_jobs=-1
)
oof_lgb = np.zeros(len(y)); test_lgb = np.zeros(len(X_te))
for f_i, (tr_idx, val_idx) in enumerate(folds):
    t0 = time.time()
    m = lgb.LGBMRegressor(**LGB_PARAMS)
    m.fit(X_tr[tr_idx], y_log[tr_idx],
          eval_set=[(X_tr[val_idx], y_log[val_idx])],
          callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)])
    oof_lgb[val_idx] = np.expm1(m.predict(X_tr[val_idx]))
    test_lgb += np.expm1(m.predict(X_te)) / N_SPLITS
    print(f"  LGB fold{f_i+1}: MAE={mean_absolute_error(y[val_idx], oof_lgb[val_idx]):.4f}  it={m.best_iteration_}  ({time.time()-t0:.0f}s)")
oof_lgb = np.clip(oof_lgb, 0, None)
np.save('results/base_v31/lgb_v31_oof.npy', oof_lgb)
np.save('results/base_v31/lgb_v31_test.npy', test_lgb)
print(f"LGB OOF MAE: {mean_absolute_error(y, oof_lgb):.5f}")

In [ ]:
# ─── XGBoost ──────────────────────────────────────────────────
XGB_PARAMS = dict(
    objective='reg:absoluteerror', n_estimators=3000, learning_rate=0.03,
    max_depth=6, min_child_weight=50, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.5, reg_lambda=0.5, tree_method='hist',
    random_state=SEED, verbosity=0, n_jobs=4, early_stopping_rounds=100
)
oof_xgb = np.zeros(len(y)); test_xgb = np.zeros(len(X_te))
for f_i, (tr_idx, val_idx) in enumerate(folds):
    t0 = time.time()
    m = xgb.XGBRegressor(**XGB_PARAMS)
    m.fit(X_tr[tr_idx], y_log[tr_idx],
          eval_set=[(X_tr[val_idx], y_log[val_idx])], verbose=False)
    oof_xgb[val_idx] = np.expm1(m.predict(X_tr[val_idx]))
    test_xgb += np.expm1(m.predict(X_te)) / N_SPLITS
    print(f"  XGB fold{f_i+1}: MAE={mean_absolute_error(y[val_idx], oof_xgb[val_idx]):.4f}  ({time.time()-t0:.0f}s)")
oof_xgb = np.clip(oof_xgb, 0, None)
np.save('results/base_v31/xgb_v31_oof.npy', oof_xgb)
np.save('results/base_v31/xgb_v31_test.npy', test_xgb)
print(f"XGB OOF MAE: {mean_absolute_error(y, oof_xgb):.5f}")

In [ ]:
# ─── CatBoost ─────────────────────────────────────────────────
oof_cb = np.zeros(len(y)); test_cb = np.zeros(len(X_te))
for f_i, (tr_idx, val_idx) in enumerate(folds):
    t0 = time.time()
    m = CatBoostRegressor(
        loss_function='MAE', iterations=3000, learning_rate=0.03,
        depth=6, min_data_in_leaf=50, subsample=0.8, rsm=0.8,
        l2_leaf_reg=3.0, random_seed=SEED, verbose=0,
        early_stopping_rounds=100, task_type='CPU', thread_count=4
    )
    m.fit(X_tr[tr_idx], y_log[tr_idx],
          eval_set=(X_tr[val_idx], y_log[val_idx]), use_best_model=True)
    oof_cb[val_idx] = np.expm1(m.predict(X_tr[val_idx]))
    test_cb += np.expm1(m.predict(X_te)) / N_SPLITS
    print(f"  CB fold{f_i+1}: MAE={mean_absolute_error(y[val_idx], oof_cb[val_idx]):.4f}  ({time.time()-t0:.0f}s)")
oof_cb = np.clip(oof_cb, 0, None)
np.save('results/base_v31/cb_v31_oof.npy', oof_cb)
np.save('results/base_v31/cb_v31_test.npy', test_cb)
print(f"CB OOF MAE: {mean_absolute_error(y, oof_cb):.5f}")

In [ ]:
# ─── Base 모델 결과 요약 ──────────────────────────────────────
base_results = {
    'LGB v31':  mean_absolute_error(y, oof_lgb),
    'XGB v31':  mean_absolute_error(y, oof_xgb),
    'CB  v31':  mean_absolute_error(y, oof_cb),
    'Simple avg': mean_absolute_error(y, (oof_lgb + oof_xgb + oof_cb) / 3),
}
for name, mae in base_results.items():
    print(f"{name:15s}: OOF MAE = {mae:.5f}")

# OOF 상관관계 확인 (앙상블 다양성)
corr_lx = np.corrcoef(oof_lgb, oof_xgb)[0,1]
corr_lc = np.corrcoef(oof_lgb, oof_cb)[0,1]
corr_xc = np.corrcoef(oof_xgb, oof_cb)[0,1]
print(f"\nOOF 상관관계 (corr >= 0.95 → 블렌드 효과 없음):")
print(f"  LGB-XGB: {corr_lx:.4f}")
print(f"  LGB-CB:  {corr_lc:.4f}")
print(f"  XGB-CB:  {corr_xc:.4f}")

## Section 5: Mega33 스태킹

### 스태킹 구조

- **Level-1 모델 (33개)**: LGB + XGB + CatBoost × 여러 피처 버전 (v23, v24, v26, v31) × 시드 변형
- **Level-2 (Meta) 모델**: LGB/XGB/CB 얕은 트리로 OOF 스택 재학습
- **최종 mega33**: 3개 meta 모델 OOF 평균

### 앙상블 다양성 원칙

> **corr ≥ 0.95 이면 블렌드 가중치 0** — 상관관계가 너무 높은 모델은 다양성에 기여하지 못함.  
> 실질적 앙상블 효과는 **0.90~0.95** 구간에서 발생.


In [ ]:
# ─── 스택 행렬 구성 ───────────────────────────────────────────
# 이전 세대 모델 OOF (v23 seeds, v24, v26, neural army 등)
# 여기서는 v31 base 3개 모델만 예시로 보여줌 (실제는 33개 모델 스택)

os.makedirs('results', exist_ok=True)

# 실제 구성: v23(LGB×4seeds + XGB×4seeds + CB×4seeds),
#            v24(LGB_tuned + XGB_tuned + CB_tuned),
#            v26(LGB + XGB + CB),
#            v31(LGB + XGB + CB),
#            neural_army(MLP×3 + TabNet×2 + 기타)
# → 총 33개 OOF 벡터

stack_oof_dict = {
    'lgb_v31': oof_lgb,
    'xgb_v31': oof_xgb,
    'cb_v31':  oof_cb,
    # ... + 30개 추가 (v23_seed42_LGB, v23_seed123_XGB, v24_Tuned_Huber, ...)
}

# log1p 변환 후 스택 (meta 모델이 log 공간에서 학습)
oof_list  = [np.log1p(np.clip(o, 0, None)) for o in stack_oof_dict.values()]
stack_train = np.column_stack(oof_list)

# test 스택 (base 모델 test 예측 동일 변환)
test_pred_list = [np.log1p(np.clip(t, 0, None)) for t in [test_lgb, test_xgb, test_cb]]
stack_test = np.column_stack(test_pred_list)

print(f"스택 shape: {stack_train.shape}")
print(f"(실제 mega33에서는 {stack_train.shape[0]} rows × 33 models)")

# 스택 내 상관관계 확인 (다양성 점검)
corr_mat = np.corrcoef(stack_train.T)
upper_tri = corr_mat[np.triu_indices_from(corr_mat, k=1)]
print(f"\n스택 모델간 상관관계: mean={upper_tri.mean():.3f}, min={upper_tri.min():.3f}, max={upper_tri.max():.3f}")
high_corr_pairs = (upper_tri >= 0.95).sum()
print(f"corr >= 0.95 인 쌍: {high_corr_pairs}개 (이 모델들은 가중치 0 처리)")

In [ ]:
# ─── Meta LGB ─────────────────────────────────────────────────
# 각 meta 모델은 스택을 입력으로 log1p(y) 예측, 얕은 트리 사용
META_LGB_PARAMS = dict(
    objective='mae', n_estimators=500, learning_rate=0.05,
    num_leaves=15, max_depth=4, min_child_samples=100,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=1.0, reg_lambda=1.0,
    random_state=SEED, verbose=-1, n_jobs=-1
)
META_XGB_PARAMS = dict(
    objective='reg:absoluteerror', n_estimators=500, learning_rate=0.05,
    max_depth=4, min_child_weight=100, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=1.0, reg_lambda=1.0, tree_method='hist',
    random_state=SEED, verbosity=0, n_jobs=4, early_stopping_rounds=50
)

oof_meta_lgb = np.zeros(len(y))
oof_meta_xgb = np.zeros(len(y))
oof_meta_cb  = np.zeros(len(y))

# GroupKFold로 meta 학습 (동일 fold 구조 재사용)
for f_i, (tr_idx, val_idx) in enumerate(folds):
    # Meta LGB
    m_lgb = lgb.LGBMRegressor(**META_LGB_PARAMS)
    m_lgb.fit(stack_train[tr_idx], y_log[tr_idx],
              eval_set=[(stack_train[val_idx], y_log[val_idx])],
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
    oof_meta_lgb[val_idx] = np.expm1(m_lgb.predict(stack_train[val_idx]))

    # Meta XGB
    m_xgb = xgb.XGBRegressor(**META_XGB_PARAMS)
    m_xgb.fit(stack_train[tr_idx], y_log[tr_idx],
               eval_set=[(stack_train[val_idx], y_log[val_idx])], verbose=False)
    oof_meta_xgb[val_idx] = np.expm1(m_xgb.predict(stack_train[val_idx]))

    # Meta CB
    m_cb = CatBoostRegressor(
        loss_function='MAE', iterations=500, learning_rate=0.05,
        depth=4, min_data_in_leaf=100, subsample=0.8, rsm=0.8,
        l2_leaf_reg=3.0, random_seed=SEED, verbose=0,
        early_stopping_rounds=50, task_type='CPU', thread_count=4
    )
    m_cb.fit(stack_train[tr_idx], y_log[tr_idx],
             eval_set=(stack_train[val_idx], y_log[val_idx]), use_best_model=True)
    oof_meta_cb[val_idx] = np.expm1(m_cb.predict(stack_train[val_idx]))

    print(f"  Meta fold{f_i+1}: LGB={mean_absolute_error(y[val_idx], oof_meta_lgb[val_idx]):.4f}  "
          f"XGB={mean_absolute_error(y[val_idx], oof_meta_xgb[val_idx]):.4f}  "
          f"CB={mean_absolute_error(y[val_idx], oof_meta_cb[val_idx]):.4f}")

# 3개 meta 평균 = mega33 OOF
mega33_oof = np.clip((oof_meta_lgb + oof_meta_xgb + oof_meta_cb) / 3, 0, None)
print(f"
mega33 OOF MAE: {mean_absolute_error(y, mega33_oof):.5f}")

# 전체 데이터로 meta 재학습 -> test 예측
_lgb_full = lgb.LGBMRegressor(**{**META_LGB_PARAMS, 'n_estimators': 300})
_lgb_full.fit(stack_train, y_log)

_xgb_p = {k: v for k, v in META_XGB_PARAMS.items() if k != 'early_stopping_rounds'}
_xgb_full = xgb.XGBRegressor(**_xgb_p, n_estimators=300)
_xgb_full.fit(stack_train, y_log)

_cb_full = CatBoostRegressor(
    loss_function='MAE', iterations=300, learning_rate=0.05,
    depth=4, min_data_in_leaf=100, subsample=0.8, rsm=0.8,
    l2_leaf_reg=3.0, random_seed=SEED, verbose=0, task_type='CPU', thread_count=4
)
_cb_full.fit(stack_train, y_log)

mega33_test = np.clip((
    np.expm1(_lgb_full.predict(stack_test)) +
    np.expm1(_xgb_full.predict(stack_test)) +
    np.expm1(_cb_full.predict(stack_test))
) / 3, 0, None)

with open('results/mega33_final.pkl', 'wb') as f:
    pickle.dump({'meta_avg_oof': mega33_oof, 'meta_avg_test': mega33_test}, f)
print("mega33 저장 완료 (oof + test)")

## Section 6: Sequential Oracle (핵심 아이디어)

### Oracle 아이디어

시계열 데이터에서 **"이전 타임슬롯의 실제 지연 시간"** 이 가장 강력한 예측 피처이다.  
그러나 test 예측 시에는 미래의 실제 값(y)을 알 수 없다.

**해결책**: mega33 OOF 예측값을 "proxy lag 피처"로 활용

```
Training:   lag1 = 진짜 y(t-1)           ← 완벽한 정보
Validation: lag1 = mega33_oof(t-1)       ← proxy (불완전하지만 유용)
Test:       lag1 = mega33_test_pred(t-1) ← 순차적으로 채워나감
```

이를 통해 **GroupKFold** 구조 내에서 sequential 패턴을 포착할 수 있다.


In [ ]:
# ─── Oracle 준비 ─────────────────────────────────────────────
train['row_in_sc'] = train.groupby(['layout_id','scenario_id']).cumcount()
test['row_in_sc']  = test.groupby(['layout_id','scenario_id']).cumcount()

y_true = train[TARGET].values
global_mean = y_true.mean()

# mega33 OOF 로드 (ls order → rid order 변환)
with open('results/mega33_final.pkl','rb') as f:
    d = pickle.load(f)
mega_oof  = d['meta_avg_oof'][id2]    # rid order (train과 동일 인덱스)

# train에 mega33 예측값 컬럼 추가
train['mega33_pred'] = mega_oof

# within-scenario true lag (training 에서만 사용 가능)
train['lag1_y'] = train.groupby(['layout_id','scenario_id'])[TARGET].shift(1).fillna(global_mean)
train['lag2_y'] = train.groupby(['layout_id','scenario_id'])[TARGET].shift(2).fillna(global_mean)
# val 에서는 mega33 proxy 사용
train['lag1_mega'] = train.groupby(['layout_id','scenario_id'])['mega33_pred'].shift(1).fillna(global_mean)
train['lag2_mega'] = train.groupby(['layout_id','scenario_id'])['mega33_pred'].shift(2).fillna(global_mean)

print(f"global_mean: {global_mean:.4f}")
print(f"lag1_y sample (first 5): {train['lag1_y'].values[:5]}")
print(f"lag1_mega sample (first 5): {train['lag1_mega'].values[:5]}")

In [ ]:
# ─── Oracle XGB 학습 ──────────────────────────────────────────
os.makedirs('results/oracle_seq', exist_ok=True)

def make_X_oracle(base_feats, lag1, lag2, row_sc):
    """base 피처 + oracle lag 피처 결합"""
    extra = np.column_stack([lag1, lag2, row_sc])
    return np.hstack([base_feats, extra])

# base 피처: X_tr (ls order) → rid order로 변환
X_base_train = X_tr[id2]   # rid order (train과 동일)

ORACLE_XGB_PARAMS = dict(
    objective='reg:absoluteerror', n_estimators=3000, learning_rate=0.05,
    max_depth=7, min_child_weight=20, subsample=0.8, colsample_bytree=0.8,
    n_jobs=4, random_state=SEED, verbosity=0, early_stopping_rounds=100
)

gkf_rid   = GroupKFold(n_splits=5)
groups_rid = train['layout_id'].values
folds_rid  = list(gkf_rid.split(np.arange(len(train)), groups=groups_rid))
oof_oracle_xgb = np.zeros(len(train))

for fold_i, (tr_idx, val_idx) in enumerate(folds_rid):
    t0 = time.time()

    # 학습: 진짜 lag y 사용
    X_tr_oracle = make_X_oracle(
        X_base_train[tr_idx],
        train['lag1_y'].values[tr_idx],
        train['lag2_y'].values[tr_idx],
        train['row_in_sc'].values[tr_idx]
    )

    # val: mega33 proxy lag 사용 (자동 회귀적으로 채워나감)
    X_val_proxy = make_X_oracle(
        X_base_train[val_idx],
        train['lag1_mega'].values[val_idx],
        train['lag2_mega'].values[val_idx],
        train['row_in_sc'].values[val_idx]
    )

    model = xgb.XGBRegressor(**ORACLE_XGB_PARAMS)
    model.fit(X_tr_oracle, y_true[tr_idx],
              eval_set=[(X_val_proxy, y_true[val_idx])],
              verbose=False)

    # val 순차 예측 (auto-regressive) — layout/scenario/position 기준 정렬
    _sort_df = pd.DataFrame({
        'lid': train['layout_id'].values[val_idx],
        'sid': train['scenario_id'].values[val_idx],
        'pos': train['row_in_sc'].values[val_idx].astype(int)
    })
    val_sorted_idx = val_idx[_sort_df.sort_values(['lid', 'sid', 'pos']).index.values]

    pred_buffer = {}   # (layout_id, scenario_id) → [예측값 리스트]
    for idx in val_sorted_idx:
        lid = train['layout_id'].values[idx]
        sid = train['scenario_id'].values[idx]
        pos = train['row_in_sc'].values[idx]
        key = (lid, sid)
        hist = pred_buffer.get(key, [])
        lag1 = hist[-1] if len(hist) >= 1 else global_mean
        lag2 = hist[-2] if len(hist) >= 2 else global_mean
        x = make_X_oracle(
            X_base_train[idx:idx+1], [lag1], [lag2], [pos]
        )
        pred_val = float(model.predict(x)[0])
        pred_val = max(0, pred_val)
        oof_oracle_xgb[idx] = pred_val
        hist.append(pred_val)
        pred_buffer[key] = hist

    mae_fold = mean_absolute_error(y_true[val_idx], oof_oracle_xgb[val_idx])
    print(f"  Oracle fold{fold_i+1}: MAE={mae_fold:.4f}  best_iter={model.best_iteration}  ({time.time()-t0:.0f}s)")

oof_oracle_xgb = np.clip(oof_oracle_xgb, 0, None)
np.save('results/oracle_seq/oof_seqC_xgb.npy', oof_oracle_xgb)
print(f"\nOracle XGB OOF MAE: {mean_absolute_error(y_true, oof_oracle_xgb):.5f}")

### Oracle 변형 (여러 버전)

| 버전 | 설명 | OOF MAE |
|------|------|---------|
| `seqC_xgb` | XGBoost oracle, auto-regressive val | 8.47 |
| `seqC_log_v2` | log 공간 oracle + 2단계 refinement | 8.51 |
| `seqC_xgb_remaining` | unseen layout 전용 oracle | 8.89 |

> Oracle 모델 단독으로는 mega33보다 약하지만, 블렌딩 시 다양성 기여로 LB 개선.


## Section 7: 최종 블렌드 (oracle_NEW)

### 블렌드 전략

최종 제출 `oracle_NEW`는 다음 5개 컴포넌트의 가중 평균:

| 컴포넌트 | 가중치 | 설명 |
|----------|--------|------|
| `mega33` | 0.7637 | 33개 모델 메타 앙상블 (주 모델) |
| `rank_adj` | 0.1589 | 순위 기반 보정 예측 |
| `iter_r1` | 0.0119 | Pseudo-labeling 1라운드 |
| `iter_r2` | 0.0346 | Pseudo-labeling 2라운드 |
| `iter_r3` | 0.0310 | Pseudo-labeling 3라운드 |

위 `fixed_oof`에 oracle_seq 3개 모델을 추가 블렌드:
- `fixed_oof × 0.64 + seqC_xgb × 0.12 + seqC_log_v2 × 0.16 + seqC_remaining × 0.08`


In [ ]:
# ─── FIXED base 계산 ─────────────────────────────────────────
fw = {
    'mega33':   0.7636614598089654,
    'rank_adj': 0.1588758398901156,
    'iter_r1':  0.011855567572749024,
    'iter_r2':  0.034568307,
    'iter_r3':  0.031038826,
}
assert abs(sum(fw.values()) - 1.0) < 1e-4, f"가중치 합: {sum(fw.values())}"

# 각 컴포넌트 로드 (ls order → rid order 변환)
rank_oof = np.load('results/ranking/rank_adj_oof.npy')[id2]
r1_oof   = np.load('results/iter_pseudo/round1_oof.npy')[id2]
r2_oof   = np.load('results/iter_pseudo/round2_oof.npy')[id2]
r3_oof   = np.load('results/iter_pseudo/round3_oof.npy')[id2]

# oracle seq 컴포넌트 (이미 rid order)
xgb_oof = np.load('results/oracle_seq/oof_seqC_xgb.npy')
lv2_oof = np.load('results/oracle_seq/oof_seqC_log_v2.npy')
rem_oof = np.load('results/oracle_seq/oof_seqC_xgb_remaining.npy')

fixed_oof = (fw['mega33']   * mega_oof  +
             fw['rank_adj'] * rank_oof  +
             fw['iter_r1']  * r1_oof    +
             fw['iter_r2']  * r2_oof    +
             fw['iter_r3']  * r3_oof)

print(f"fixed_oof OOF MAE: {mean_absolute_error(y_true, np.clip(fixed_oof, 0, None)):.5f}")

In [ ]:
# ─── oracle_NEW 최종 블렌드 ───────────────────────────────────
oracle_oof = np.maximum(0,
    0.64 * fixed_oof +
    0.12 * xgb_oof   +
    0.16 * lv2_oof   +
    0.08 * rem_oof
)
print(f"oracle_NEW OOF MAE: {mean_absolute_error(y_true, oracle_oof):.5f}")
# → 약 8.38247

# seen/unseen 별 성능 확인
seen_mask   = train['layout_id'].isin(seen_test)
unseen_mask = train['layout_id'].isin(unseen_test)
print(f"  seen   MAE: {mean_absolute_error(y_true[seen_mask],   oracle_oof[seen_mask]):.5f}")
print(f"  unseen MAE: {mean_absolute_error(y_true[unseen_mask], oracle_oof[unseen_mask]):.5f}")

In [ ]:
# ─── 제출 파일 생성 ───────────────────────────────────────────
# (test 예측도 동일한 가중치 적용)
with open('results/mega33_final.pkl', 'rb') as _f:
    d_final = pickle.load(_f)
mega_test = d_final['meta_avg_test'][te_id2]
rank_test = np.load('results/ranking/rank_adj_test.npy')[te_id2]
r1_test   = np.load('results/iter_pseudo/round1_test.npy')[te_id2]
r2_test   = np.load('results/iter_pseudo/round2_test.npy')[te_id2]
r3_test   = np.load('results/iter_pseudo/round3_test.npy')[te_id2]
xgb_test  = np.load('results/oracle_seq/test_C_xgb.npy')
lv2_test  = np.load('results/oracle_seq/test_C_log_v2.npy')
rem_test  = np.load('results/oracle_seq/test_C_xgb_remaining.npy')

fixed_test = (fw['mega33']   * mega_test +
              fw['rank_adj'] * rank_test +
              fw['iter_r1']  * r1_test   +
              fw['iter_r2']  * r2_test   +
              fw['iter_r3']  * r3_test)

oracle_test = np.maximum(0,
    0.64 * fixed_test +
    0.12 * xgb_test   +
    0.16 * lv2_test   +
    0.08 * rem_test
)

sub = pd.DataFrame({
    'ID': sample['ID'],
    'avg_delay_minutes_next_30m': oracle_test
})
sub.to_csv('submission_oracle_NEW.csv', index=False)
print(f"제출 파일 저장: submission_oracle_NEW.csv")
print(f"예측값 통계: mean={oracle_test.mean():.3f}, std={oracle_test.std():.3f}, min={oracle_test.min():.3f}, max={oracle_test.max():.3f}")
print(f"\n→ Public LB: 9.7527 (17위 / 608팀)")

## Section 8: 삽질 기록 & 교훈

### 시도했지만 실패한 접근들

#### 1. Gate 계열: seen/unseen 분기 최적화

**아이디어**: test_seen과 test_unseen에 서로 다른 가중치를 적용  
**결과**: OOF에서 개선되어 보이지만 **LB에서 악화**  
**교훈**: test set의 unseen layout 비율을 train OOF로 정확히 시뮬레이션할 수 없음. 분기 최적화 자체가 test 분포에 과적합하는 형태.

#### 2. Scipy 최적화: 가중치 탐색

**아이디어**: `scipy.optimize.minimize`로 OOF 기준 최적 블렌드 가중치 탐색  
**결과**: OOF에 과적합 (OOF MAE 0.05 개선 → LB 0.02 악화)  
**교훈**: 모델 수가 많을수록 scipy 최적화가 OOF에 과적합. 대신 상관관계 기반 다양성 점검 후 수동 가중치 설정이 더 안정적.

#### 3. Temporal CV: 시간 순서 기반 교차검증

**아이디어**: 시나리오 내 시간 순서를 반영한 temporal GroupKFold  
**결과**: OOF=8.63 / 상관관계 corr=0.9571 → 모든 블렌드에서 음수 기여  
**교훈**: **GroupKFold oracle이 실제로 더 잘 일반화됨.** Temporal CV는 val에 unseen layout이 적어 낙관적 bias 발생.

#### 4. test_unseen 보정: 별도 offset 추가

**아이디어**: unseen layout 예측값에 global_mean 기반 보정 적용  
**결과**: tw11 + packOOD 실증 → **LB 악화**  
**교훈**: test 분포를 정확히 알 수 없는 상황에서 단방향 보정은 위험. unseen 보정이 seen 예측에 노이즈를 추가.

#### 5. Neural Army: MLP/TabNet 앙상블

**아이디어**: GBDT와 다른 귀납 편향으로 다양성 확보  
**결과**: 단독 성능은 낮지만 GBDT corr=0.91~0.93 → 소폭 기여  
**교훈**: **corr ≥ 0.95이면 블렌드 가중치 0.** 실질적 앙상블 sweet spot은 corr 0.90~0.95.

#### 6. GP (Gaussian Process) 보정

**시도**: OOF 잔차를 GP로 모델링해서 test에 적용  
**결과**: gain=0 또는 LB 악화  
**교훈**: 잔차 패턴이 test에서 재현되지 않음 (test 분포 차이).

---

### 핵심 교훈 요약

1. **OOF 개선 ≠ LB 개선**: 특히 seen/unseen 분기나 scipy 최적화처럼 test 분포를 가정하는 방법은 항상 LB에서 검증 필요
2. **GroupKFold(layout_id)가 최선**: 시뮬레이션 데이터 특성상, layout 기준 CV가 실제 generalization을 가장 잘 반영
3. **다양성 > 정확도**: 개별 모델 성능보다 OOF 상관관계가 낮은 모델 조합이 앙상블에 유리
4. **남은 제출권 보존**: 이미 ceiling에 도달했다고 판단 시 무분별한 제출보다 분석이 중요
5. **Sequential Oracle 효과**: mega33 OOF를 proxy lag으로 활용하는 전략이 시계열 패턴 포착에 유효 (단, 단독보다 블렌딩으로)


## 결론

| 단계 | OOF MAE | Public LB |
|------|---------|----------|
| Base LGB/XGB/CB 단순 평균 | ~9.1 | ~9.95 |
| Mega33 스태킹 | ~8.55 | ~9.82 |
| + Pseudo-labeling | ~8.48 | ~9.78 |
| + Sequential Oracle 블렌드 | **8.3825** | **9.7527** |

**최종 순위**: 608팀 중 **17위** (Public 9.7527 / Private 10.01261)

코드 공유나 질문은 데이콘 토론 게시판에서 환영합니다!
